# Export final results to Excel — gemma4:e4b + pdfplumber

The pipeline's **final output** for every document under the `gemma4-e4b_pdfplumber_whole_doc` tag lives in
`data/output/4_final/` — the structured DMP JSON after the annotation rules, the
file a user of the package receives. This notebook flattens each one into a
two-column **(label, text)** table in document order and writes them all to one
Excel workbook:

- `data/output/pdfplumber_fallback_lightonocr_and_gemma.xlsx`
- one sheet per sample

Because this reads stage 4, every sheet matches the final JSON exactly —
including what the converter merged and what the rules filled in. (The model's
raw pre-structure blocks live in `2_labeled/` if those are ever needed instead.)


In [1]:
import os
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

import json

import pandas as pd
from IPython.display import display

TAG = 'gemma4-e4b_pdfplumber_whole_doc'
FINAL_DIR = Path('data/output/4_final') / TAG
XLSX = Path('data/output/pdfplumber_fallback_lightonocr_and_gemma.xlsx')

pd.set_option('display.max_colwidth', 90)


## 1. The flattening function

One function, one job: a final structured JSON in, a tidy `(label, text)` table
out, rows in document order — the document title first, then each section's
title, description, questions and answers. Empty fields produce no row, so the
table holds exactly what the final JSON holds.


In [2]:
def flatten_final_json(path: Path) -> pd.DataFrame:
    """Flatten one final structured DMP JSON into a (label, text) table."""
    data = json.loads(path.read_text(encoding='utf-8'))
    template = data.get('narrative', {}).get('template')
    if template is None:
        raise ValueError(f'{path.name}: not a structured DMP JSON (no narrative.template)')
    rows = []

    def add(label, text):
        if text and text.strip():
            rows.append({'label': label, 'text': text.strip()})

    add('title', template.get('title', ''))
    for section in template.get('section', []):
        add('section.title', section.get('title', ''))
        add('section.description', section.get('description', ''))
        for q in section.get('question', []):
            add('question.text', q.get('text', ''))
            add('answer.text', q.get('answer', {}).get('json', {}).get('answer', ''))
    return pd.DataFrame(rows, columns=['label', 'text'])


## 2. Flatten every sample under the tag

The summary shows, per document, how many rows the final JSON yields and how
they split across the five labels.


In [3]:
files = sorted(FINAL_DIR.glob('sample*.json'),
               key=lambda p: int(p.stem.replace('sample', '')))
tables = {p.stem: flatten_final_json(p) for p in files}

summary = pd.DataFrame([
    {'sample': name, 'rows': len(df), **df['label'].value_counts().to_dict()}
    for name, df in tables.items()
]).fillna(0)
for col in summary.columns:
    if col not in ('sample',):
        summary[col] = summary[col].astype(int)
display(summary.set_index('sample'))


,rows,answer.text,question.text,section.title,title,section.description
sample,,,,,,
sample1,31,12,12,6,1,0
sample2,23,7,7,4,1,4
sample3,19,4,5,5,1,4
sample4,4,1,1,1,1,0
sample5,20,5,6,6,1,2
sample6,16,5,5,5,1,0
sample7,4,1,1,1,1,0
sample8,19,6,6,6,1,0
sample9,16,5,5,5,1,0


One table up close — the format every sheet in the workbook uses.


In [4]:
display(tables['sample13'].head(10))


,label,text
0,title,Understanding the protein composition and dynamics of the nuclear envelope in plants
1,section.title,DATA MANAGEMENT PLAN
2,question.text,DATA MANAGEMENT PLAN
3,answer.text,The Gu Laboratory is committed to the open sharing and prompt dissemination of all dat...
4,section.title,Proteomic Data
5,section.description,This project will generate PL-LFQMS profiling data using 13 newly identified PNET prot...
6,question.text,Proteomic Data
7,answer.text,We will make the raw proteomic data available to the public as soon as it has passed q...
8,section.title,"Mutants, Transgenic Lines, and Plasmid Constructs"
9,section.description,"This project will generate biological materials of use to the research community, incl..."


## 3. Export to Excel

One sheet per sample, two columns each.


In [5]:
XLSX.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(XLSX, engine='openpyxl') as writer:
    for name, df in tables.items():
        df.to_excel(writer, sheet_name=name, index=False)
    for sheet in writer.sheets.values():
        sheet.column_dimensions['A'].width = 22
        sheet.column_dimensions['B'].width = 130

print(f'wrote {XLSX}  ({XLSX.stat().st_size / 1024:.0f} KB)')
print(f'{len(tables)} sample sheets, {sum(len(df) for df in tables.values())} rows total')


wrote data\output\pdfplumber_fallback_lightonocr_and_gemma.xlsx  (44 KB)
13 sample sheets, 216 rows total
